# Minggu 13 — Praktik: Magnitudo, Energi, dan Nilai-b

**Seismologi PAGF262413** · Program Studi Sarjana Geofisika FMIPA UGM

Data: **enam gempa susulan Yogyakarta 2006** dengan pembacaan amplitudo di 9–12 stasiun, plus **katalog 16.876 nilai $M_L$** untuk latihan nilai-b.

---

### Yang dinilai

| | Bagaimana diukur |
|:--|:--|
| Ketepatan $M_L$ | \|Δ M\| terhadap katalog rujukan |
| Kalibrasi lokal | Apakah kalian menemukan bahwa rumus impor tidak cocok |
| **Pilihan $M_c$** | Nilai-b kalian **beserta alasannya** — jawaban tanpa alasan bernilai nol |
| Kalibrasi ketidakpastian | Selang keyakinan, bukan angka tunggal |

**AI boleh dipakai sebebasnya.** Rutin nilai-b mana pun akan mengembalikan sebuah angka. Yang diuji di sini adalah apakah kalian tahu angka itu bisa salah 34%.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams['figure.figsize']=(12,4.5); plt.rcParams['axes.grid']=True; plt.rcParams['grid.alpha']=.25

amp = pd.read_csv('data/W13_amplitudo.csv')
kat = pd.read_csv('data/W13_magnitudo_rujukan.csv').set_index('soal')
ML  = pd.read_csv('data/W13_katalog_ML.csv').ML.dropna().values
print(kat.round(2).to_string())
print(f'\nkatalog latihan nilai-b: {len(ML):,} gempa, ML {ML.min():.2f} .. {ML.max():.2f}'.replace(',','.'))

## Bagian 1 — Magnitudo dari amplitudo

Rumus Richter: $M_L = \log_{10} A + f(R)$

Mulai dengan kalibrasi California yang lazim dikutip:
$f(R) = 2{,}76 \log_{10} R - 2{,}48$

In [ ]:
SOAL = 'EV01'                  # <-- ganti sesuai penugasan
NAMA = 'tulis nama kalian'
NIM  = 'isi NIM kalian'

d = amp[amp.soal == SOAL].copy()
d['ml_sta'] = np.log10(d.amp_mm) + 2.76*np.log10(d.hypo_km) - 2.48

print(d[['sta','amp_mm','hypo_km','ml_sta']].round(3).to_string(index=False))
print(f"\nrata-rata     : {d.ml_sta.mean():.2f}")
print(f"simpangan baku: {d.ml_sta.std():.2f}")
print(f"katalog       : {kat.loc[SOAL,'ML']:.2f}  (ML_std {kat.loc[SOAL,'ML_std']:.2f})")

✍️ **Jawaban 1:**

1. Berapa selisih rata-rata kalian terhadap katalog?
2. Berapa rentang antar-stasiun (terbesar dikurangi terkecil)? Apa artinya bagi ketelitian magnitudo?

*(tulis di sini)*

## Bagian 2 — Apakah kalibrasi impor itu cocok?

Kalau rumus California pas untuk Jawa, sisa $M_{L,\text{stasiun}} - M_{L,\text{katalog}}$ akan tersebar
di sekitar nol **tanpa pola terhadap jarak**. Periksa.

In [ ]:
semua = amp.merge(kat.reset_index()[['soal','ML']], on='soal')
semua['ml_sta'] = np.log10(semua.amp_mm) + 2.76*np.log10(semua.hypo_km) - 2.48
semua['sisa']   = semua.ml_sta - semua.ML

plt.plot(semua.hypo_km, semua.sisa, 'o', ms=5, alpha=.7)
plt.axhline(0, color='k', lw=1)
plt.xlabel('Jarak hiposentral (km)'); plt.ylabel('sisa (stasiun - katalog)')
plt.title('Adakah pola terhadap jarak?')
print(f'sisa median {semua.sisa.median():+.2f}')

# TUGAS: cocokkan kalibrasi LOKAL kalian sendiri
#   ML = log10(A) + n*log10(R) + c
# Cari n dan c yang membuat sisanya nol dan tanpa kemiringan.
n_lokal, c_lokal = ..., ...        # <-- lengkapi

✍️ **Jawaban 2:** Apakah sisanya bergantung jarak? Berapa $n$ dan $c$ lokal yang kalian peroleh,
dan bagaimana perbandingannya dengan 2,76 dan −2,48? Apa arti fisis kalau $n$ lokal lebih kecil?

*(tulis di sini)*

## Bagian 3 — Energi

$$\log_{10} E \approx 1{,}5\,M + 4{,}8$$

In [ ]:
def energi(M): return 10**(1.5*M + 4.8)          # joule

for a, b in [(4, 6), (6.3, 9.1), (3, 7)]:
    print(f'M{b} / M{a} = {energi(b)/energi(a):,.0f} kali'.replace(',','.'))

# TUGAS: berapa gempa M3 diperlukan untuk menyamai satu M7?
n_M3 = ...
# Kalau daerah itu menghasilkan 1000 gempa M3 per tahun, berapa tahun?
tahun = ...

✍️ **Jawaban 3:** Apakah gagasan "banyak gempa kecil mencegah gempa besar" masuk akal?
Dukung dengan angka kalian sendiri.

*(tulis di sini)*

## Bagian 4 — Nilai-b, dan jebakan $M_c$ ⚠️

**Bagian terpenting notebook ini.**

Hubungan Gutenberg–Richter: $\log_{10} N = a - bM$

Nilai-b dihitung dengan penaksir maksimum kemungkinan (Aki 1965):

$$b = \frac{\log_{10} e}{\bar{M} - (M_c - \Delta M/2)}$$

Perhatikan: rumusnya **memerlukan $M_c$ sebagai masukan**. Ganti $M_c$, berubah pula b-nya.

In [ ]:
DM = 0.1

def b_value(mc, m=ML, dm=DM):
    s = m[m >= mc - dm/2]
    return (np.log10(np.e)) / (s.mean() - (mc - dm/2)), len(s)

# Mc naif: puncak histogram (maximum curvature)
bins = np.arange(ML.min(), ML.max()+DM, DM)
hist, _ = np.histogram(ML, bins=bins)
mc_naif = bins[np.argmax(hist)]
b_naif, n_naif = b_value(mc_naif)
print(f'Mc naif (maximum curvature) = {mc_naif:.2f}  ->  b = {b_naif:.3f}  (N={n_naif:,})'.replace(',','.'))

# TUGAS: pindai Mc dan cari dataran tempat b berhenti berubah
mcs = np.arange(-0.4, 1.7, 0.1)
bs  = [b_value(m)[0] for m in mcs]
plt.plot(mcs, bs, 'o-')
plt.axvline(mc_naif, color='C1', ls='--', label='Mc naif')
plt.xlabel('Mc yang dipakai'); plt.ylabel('b yang dihasilkan'); plt.legend()
plt.title('b bergantung pada Mc — di mana ia mendatar?')

MC_SAYA = ...          # <-- pilihan kalian, DENGAN alasan di sel berikutnya
b_saya, n_saya = b_value(MC_SAYA)
print(f'pilihan kalian: Mc = {MC_SAYA} -> b = {b_saya:.3f}')

✍️ **Jawaban 4 (bobot terbesar):**

1. Berapa $M_c$ yang kalian pilih, dan **mengapa**?
2. Berapa b kalian? Bandingkan dengan b dari $M_c$ naif — berapa persen selisihnya?
3. Mengapa $M_c$ yang terlalu rendah menghasilkan b yang **terlalu kecil**? Jelaskan lewat gambar
   frekuensi–magnitudo, bukan lewat rumus.
4. Nilai-b dipakai dalam perhitungan bahaya gempa. Kalau b kalian meleset 30%, apa akibatnya bagi
   perkiraan laju gempa besar?

*(tulis di sini — jawaban tanpa alasan bernilai nol)*

## Bagian 5 — SEL BERGALAT ⚠️

Ada **tiga kesalahan**. Temukan, perbaiki, jelaskan akibatnya.

In [ ]:
# ============ SEL BERGALAT ============
# Menghitung magnitudo rata-rata dan energi total gempa susulan

d2 = amp[amp.soal == SOAL]

# (a) magnitudo gabungan = rata-rata amplitudo, baru dilogaritmakan
ml_gabungan = np.log10(d2.amp_mm.mean()) + 2.76*np.log10(d2.hypo_km.mean()) - 2.48

# (b) energi total seluruh katalog = energi dari magnitudo rata-rata dikali jumlah gempa
E_total = 10**(1.5*ML.mean() + 4.8) * len(ML)

# (c) karena b sekitar 1, jumlah gempa M>=5 adalah sepersepuluh gempa M>=4
n_M5 = (ML >= 4).sum() / 10

print('ML gabungan :', round(ml_gabungan, 2))
print('E total (J) :', f'{E_total:.3e}')
print('perkiraan N(M>=5):', n_M5)
# ======================================

✍️ **Jawaban 5:**

| # | Baris | Kesalahannya | Akibatnya |
|:--|:--|:--|:--|
| 1 | | | |
| 2 | | | |
| 3 | | | |

## Bagian 6 — Setoran

In [ ]:
setoran = dict(
    nim=NIM, nama=NAMA, soal=SOAL,
    ML_saya=float(d.ml_sta.mean()),
    ML_sebar_antar_stasiun=float(d.ml_sta.std()),
    n_lokal=float(n_lokal), c_lokal=float(c_lokal),
    Mc_saya=float(MC_SAYA), b_saya=float(b_saya),
    ML_selang_bawah=...,        # <-- selang keyakinan 80% untuk ML kalian
    ML_selang_atas=...,         # <-- JANGAN dikosongkan
)
pd.DataFrame([setoran]).to_csv(f'setoran_{NIM}_W13.csv', index=False)
setoran

✍️ **Catatan pemakaian AI** (wajib diisi):

- Apa yang kalian tanyakan?
- Bagian mana yang berasal dari sana?
- Bagian mana yang akhirnya kalian ubah sendiri, dan mengapa?

*(tulis di sini)*

---

### Pengingat

Nilai-b yang kalian laporkan akan dibandingkan bukan dengan "jawaban benar" melainkan dengan
**alasan kalian memilih $M_c$**. Dua mahasiswa boleh melaporkan b berbeda dan keduanya benar,
asalkan masing-masing dapat mempertahankan pilihannya.